# Шаг 2. Поиск кандидатов: BM25 + векторный поиск

Здесь я строю первую ступень поиска. Её задача - не идеально отранжировать статьи,
а быстро отобрать для каждого запроса **топ-50 кандидатов** так, чтобы правильные ответы
почти наверняка оказались внутри. Точную расстановку внутри полусотни сделают следующие,
более тяжёлые ступени (шаги 3-4).

Использую два принципиально разных поиска и складываю их:

- **BM25** - классический лексический (словарный) поиск: статья тем релевантнее, чем больше
  в ней редких слов из запроса. Работает точно, когда пользователь употребил те же слова,
  что и статья, и бессилен против синонимов и опечаток.
- **Векторный (dense) поиск** - нейросеть-эмбеддер превращает текст в вектор чисел так,
  что близкие по смыслу тексты получают близкие векторы. Он ловит «отправить кроссовки» ≈
  «доставка товара», но иногда промахивается в деталях, где лексика точнее.

Их сумма с весами (**fusion**) стабильно лучше каждого по отдельности - ниже это видно по метрикам.

Запуск: Colab с GPU (Runtime → Change runtime type -> **T4 GPU**). С нуля ~20–30 минут,
при повторном запуске эмбеддинги берутся из кэша на Диске и всё занимает пару минут.

## 0. Установка зависимостей

In [1]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--upgrade-strategy", "only-if-needed",
        "transformers>=4.50,<5", "sentence-transformers>=4,<6", "accelerate>=1,<2",
        "huggingface-hub>=0.28,<1", "rank-bm25==0.2.2",
        "scikit-learn>=1.4,<2", "pyarrow", "tqdm",
    ])
    print("Зависимости установлены.")
else:
    print("Не Colab: зависимости из requirements.txt.")

Зависимости установлены.


## 1. Google Диск и артефакты шага 1

In [3]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/avito_rag")
else:
    PROJECT_ROOT = Path.cwd().resolve().parent / "avito_rag"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EMBEDDINGS_DIR = ARTIFACTS_DIR / "embeddings"
CANDIDATES_DIR = ARTIFACTS_DIR / "candidates"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

required = [
    ARTIFACTS_DIR / "articles_processed.parquet",
    ARTIFACTS_DIR / "calibration_processed.parquet",
    ARTIFACTS_DIR / "test_processed.parquet",
    ARTIFACTS_DIR / "article_chunks_384_58.parquet",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Сначала выполните ноутбук 01 — не найдены артефакты:\n" + "\n".join(missing)
    )
print("PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/avito_rag


In [4]:
from __future__ import annotations

import gc
import hashlib
import json
import re
import time
import unicodedata
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

RANDOM_STATE = 42
TOP_K = 10
CANDIDATE_N = 50          # сколько кандидатов оставляю на запрос
EMBEDDER_NAME = "BAAI/bge-m3"

QUERY_MAX_LENGTH = 512    # максимум токенов при кодировании запроса
CHUNK_MAX_LENGTH = 384    # чанки нарезаны под этот размер в шаге 1
DENSE_WEIGHT = 0.75       # вес векторного поиска в fusion
BM25_WEIGHT = 0.25        # вес BM25 в fusion

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUERY_BATCH_SIZE = 64 if DEVICE == "cuda" else 16
CHUNK_BATCH_SIZE = 12 if DEVICE == "cuda" else 4

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 60)

print("Device:", DEVICE)
if DEVICE != "cuda":
    print("Внимание: GPU не найден, кодирование эмбеддингов будет очень медленным.")

articles = pd.read_parquet(ARTIFACTS_DIR / "articles_processed.parquet")
calibration = pd.read_parquet(ARTIFACTS_DIR / "calibration_processed.parquet")
test = pd.read_parquet(ARTIFACTS_DIR / "test_processed.parquet")
chunks = pd.read_parquet(ARTIFACTS_DIR / "article_chunks_384_58.parquet")

article_ids = articles["article_id"].astype(np.int64).to_numpy()
article_id_to_title = articles.set_index("article_id")["clean_title"].astype(str).to_dict()
article_id_to_column = {int(a): i for i, a in enumerate(article_ids)}

print("articles:", articles.shape, "| chunks:", chunks.shape)
print("calibration:", calibration.shape, "| test:", test.shape)

Device: cuda
articles: (793, 7) | chunks: (3787, 5)
calibration: (500, 5) | test: (500, 3)


## 2. Нормализация текста для поиска

Единая функция нормализации для запросов и статей: юникод к канонической форме (NFKC),
нижний регистр, плейсхолдеры вида `<date>` привожу к `<DATE>` (в статьях встречаются такие
служебные подстановки), пунктуацию отделяю пробелами, чтобы «проверке,почему» распалось
на отдельные слова. Лемматизацию и удаление стоп-слов сознательно **не** делаю: BM25 с ними
на этих данных не выигрывал.

In [5]:
PLACEHOLDER_RE = re.compile(r"<\s*([A-Za-z][A-Za-z0-9_]*)\s*>")
PUNCT_TO_SEPARATE_RE = re.compile(r"([,.;:!?()\[\]{}«»\"/\\])")
TOKEN_RE = re.compile(r"<[A-Z][A-Z0-9_]*>|[A-Za-zА-Яа-яЁё0-9]+", re.UNICODE)


def normalize_retrieval_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\u00A0", " ").lower()
    text = PLACEHOLDER_RE.sub(lambda match: f"<{match.group(1).upper()}>", text)
    text = PUNCT_TO_SEPARATE_RE.sub(r" \1 ", text)
    return re.sub(r"\s+", " ", text).strip()


def lexical_tokenize(value: Any) -> list[str]:
    return TOKEN_RE.findall(normalize_retrieval_text(value))


def normalize_scores_per_query(scores: np.ndarray) -> np.ndarray:
    """Min-max нормализация каждой строки (запроса) в [0, 1] — чтобы складывать разные шкалы."""
    scores = np.asarray(scores, dtype=np.float32)
    shifted = scores - scores.min(axis=1, keepdims=True)
    row_max = shifted.max(axis=1, keepdims=True)
    return np.divide(shifted, row_max, out=np.zeros_like(shifted), where=row_max > 0)


def stable_hash(values: Iterable[Any]) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\x00")
    return digest.hexdigest()


articles["title_norm"] = articles["clean_title"].map(normalize_retrieval_text)
articles["body_norm"] = articles["clean_body"].map(normalize_retrieval_text)
calibration["query_norm"] = calibration["raw_query_text"].map(normalize_retrieval_text)
test["query_norm"] = test["raw_query_text"].map(normalize_retrieval_text)

assert calibration["query_norm"].str.strip().ne("").all()
assert test["query_norm"].str.strip().ne("").all()
print("Нормализация готова.")

Нормализация готова.


## 3. Метрики качества

Основная метрика задачи — **MAP@10** (Mean Average Precision). Для одного запроса считается
AP@10: иду по моему топ-10 сверху вниз, на каждой позиции с релевантной статьёй беру точность
на этой глубине (сколько релевантных нашлось к этому моменту / номер позиции), суммирую
и делю на число релевантных. Затем усредняю по всем запросам.

На пальцах: если правильные статьи `{1909, 4396}`, а я вернул `[1909, 4403, 4396, ...]`,
то AP = (1/1 + 2/3) / 2 ≈ 0.83. Чем выше стоят релевантные - тем больше вклад, и учитываются
**все** релевантные, а не только первая. Именно поэтому важно было понять на шаге 1,
что у трети запросов правильных статей несколько.

Вспомогательно смотрю ещё три метрики: Recall@10 (какая доля релевантных попала в топ-10),
MRR@10 (насколько высоко стоит первая релевантная), HitRate@10 (есть ли хоть одна релевантная в топ-10).

In [6]:
import math


def unique_top_k(prediction: Iterable[int], k: int) -> list[int]:
    result, seen = [], set()
    for value in prediction:
        article_id = int(value)
        if article_id in seen:
            continue
        seen.add(article_id)
        result.append(article_id)
        if len(result) >= k:
            break
    return result


def average_precision_at_k(ground_truth: Iterable[int], prediction: Iterable[int], k: int = TOP_K) -> float:
    relevant = set(int(x) for x in ground_truth)
    if not relevant:
        return 0.0
    hits, precision_sum = 0, 0.0
    for rank, article_id in enumerate(unique_top_k(prediction, k), start=1):
        if article_id in relevant:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / min(len(relevant), k)


def map_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    return float(np.mean([average_precision_at_k(gt, pred, k) for gt, pred in zip(ground_truths, predictions)]))


def recall_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    values = []
    for gt, pred in zip(ground_truths, predictions):
        relevant = set(int(x) for x in gt)
        values.append(len(relevant.intersection(pred[:k])) / len(relevant) if relevant else 0.0)
    return float(np.mean(values))


def mrr_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    values = []
    for gt, pred in zip(ground_truths, predictions):
        relevant = set(int(x) for x in gt)
        rank = next((i for i, a in enumerate(pred[:k], start=1) if a in relevant), None)
        values.append(1.0 / rank if rank else 0.0)
    return float(np.mean(values))


def hit_rate_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    return float(np.mean([bool(set(gt).intersection(pred[:k])) for gt, pred in zip(ground_truths, predictions)]))


def evaluate_rankings(query_table: pd.DataFrame, rankings: list[list[int]]) -> dict[str, float]:
    truths = query_table["ground_truth"].tolist()
    return {
        "MAP@10": map_at_k(truths, rankings),
        "Recall@10": recall_at_k(truths, rankings),
        "MRR@10": mrr_at_k(truths, rankings),
        "HitRate@10": hit_rate_at_k(truths, rankings),
    }


# Мини-тест на примере из текста выше.
assert math.isclose(average_precision_at_k([1909, 4396], [1909, 4403, 4396]), (1.0 + 2.0 / 3.0) / 2.0)
print("Метрики определены и проверены.")

Метрики определены и проверены.


## 4. Разбиение calibration: development / holdout

500 размеченных запросов - это всё, что у меня есть для настройки. Чтобы не обмануть самого
себя, делю их один раз и навсегда: **development (400)** - на нём выбираю модели и параметры,
**holdout (100)** - отложенная часть, которую трогаю только для финальной проверки уже
принятых решений.

Тонкость: в calibration встречаются почти одинаковые запросы (вплоть до дублей после
нормализации). Если такой дубль попадёт одной копией в development, а другой в holdout,
проверка станет нечестной - модель «видела ответ». Поэтому разбиваю не запросы, а **группы**
одинаковых нормализованных запросов, со стратификацией по числу релевантных статей
(чтобы в обеих частях была одинаковая доля «многоответных» запросов).

In [7]:
from sklearn.model_selection import StratifiedShuffleSplit


def ensure_id_list(value: Any) -> list[int]:
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    if isinstance(value, str):
        return [int(token) for token in re.findall(r"\d+", value)]
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return list(dict.fromkeys(int(item) for item in value))
    return [int(value)]


calibration["ground_truth"] = calibration["ground_truth_ids"].map(ensure_id_list)
assert calibration["ground_truth"].map(len).gt(0).all()

calibration["relevant_count"] = calibration["ground_truth"].map(len)
calibration["stratum"] = pd.cut(
    calibration["relevant_count"], bins=[0, 1, 2, np.inf], labels=["1", "2", "3+"]
).astype(str)

group_table = calibration.groupby("query_norm", as_index=False).agg(
    stratum=("stratum", lambda values: values.value_counts().index[0]),
    row_count=("query_id", "size"),
)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
dev_group_idx, holdout_group_idx = next(splitter.split(group_table["query_norm"], group_table["stratum"]))
dev_groups = set(group_table.iloc[dev_group_idx]["query_norm"])
holdout_groups = set(group_table.iloc[holdout_group_idx]["query_norm"])
calibration["split"] = np.where(calibration["query_norm"].isin(dev_groups), "development", "holdout")

assert dev_groups.isdisjoint(holdout_groups)
assert int(calibration["split"].eq("development").sum()) == 400
assert int(calibration["split"].eq("holdout").sum()) == 100

split_manifest = calibration[["query_id", "split"]].copy()
split_manifest.to_parquet(ARTIFACTS_DIR / "calibration_split.parquet", index=False)
print("Разбиение зафиксировано:", calibration["split"].value_counts().to_dict())
print("Групп повторяющихся запросов:", int(group_table["row_count"].gt(1).sum()))

Разбиение зафиксировано: {'development': 400, 'holdout': 100}
Групп повторяющихся запросов: 0


## 5. BM25

Считаю BM25 отдельно по заголовкам и отдельно по телам статей, нормализую каждый скор
per-query в [0, 1] и складываю с весом заголовка 0.5. Раздельный подсчёт нужен, потому что
заголовок - короткий концентрированный сигнал: совпадение слова с заголовком значит больше,
чем то же слово где-то в длинном тексте, и вес позволяет этим управлять. Вес 0.5 выбран
перебором на development (пробовал 0.5/1/2/3 - подробности в README).

In [8]:
from rank_bm25 import BM25Okapi

title_tokens = articles["title_norm"].map(lexical_tokenize).tolist()
body_tokens = articles["body_norm"].map(lexical_tokenize).tolist()

bm25_title = BM25Okapi(title_tokens)
bm25_body = BM25Okapi(body_tokens)

BM25_TITLE_WEIGHT = 0.5


def bm25_score_matrix(query_texts: list[str], desc: str) -> np.ndarray:
    query_tokens = [lexical_tokenize(text) for text in query_texts]
    title_scores = np.vstack([
        bm25_title.get_scores(tokens) for tokens in tqdm(query_tokens, desc=f"BM25 title ({desc})")
    ]).astype(np.float32)
    body_scores = np.vstack([
        bm25_body.get_scores(tokens) for tokens in tqdm(query_tokens, desc=f"BM25 body ({desc})")
    ]).astype(np.float32)
    return (
        BM25_TITLE_WEIGHT * normalize_scores_per_query(title_scores)
        + normalize_scores_per_query(body_scores)
    ).astype(np.float32)


bm25_calibration_scores = bm25_score_matrix(calibration["query_norm"].astype(str).tolist(), "calibration")
bm25_test_scores = bm25_score_matrix(test["query_norm"].astype(str).tolist(), "test")

assert bm25_calibration_scores.shape == (len(calibration), len(articles))
assert bm25_test_scores.shape == (len(test), len(articles))
print("BM25 матрицы:", bm25_calibration_scores.shape, bm25_test_scores.shape)

BM25 title (calibration):   0%|          | 0/500 [00:00<?, ?it/s]

BM25 body (calibration):   0%|          | 0/500 [00:00<?, ?it/s]

BM25 title (test):   0%|          | 0/500 [00:00<?, ?it/s]

BM25 body (test):   0%|          | 0/500 [00:00<?, ?it/s]

BM25 матрицы: (500, 793) (500, 793)


## 6. Эмбеддинги BGE-M3

Кодирую моделью `BAAI/bge-m3` все чанки статей и все запросы. Эмбеддинги нормированы
(длина вектора 1), поэтому скалярное произведение векторов - это косинусная близость:
число от -1 до 1, чем больше, тем ближе тексты по смыслу.

Кодирование чанков - самая дорогая операция ноутбука, поэтому результат кэшируется на Диск
вместе с метаданными (модель, хэши текстов и идентификаторов). При повторном запуске кэш
переиспользуется, а при любом изменении входных текстов - автоматически пересчитывается.

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDER_NAME, device=DEVICE,
                            model_kwargs={"torch_dtype": torch.float16 if DEVICE == "cuda" else torch.float32})
print("Эмбеддер загружен:", EMBEDDER_NAME)


def encode_or_load(cache_stem: str, texts: list[str], ids: list[Any], max_length: int, batch_size: int) -> np.ndarray:
    array_path = EMBEDDINGS_DIR / f"{cache_stem}.npy"
    metadata_path = EMBEDDINGS_DIR / f"{cache_stem}.metadata.json"
    expected = {
        "model_name": EMBEDDER_NAME, "text_count": len(texts), "max_length": int(max_length),
        "normalization": "L2; sentence-transformers normalize_embeddings=True",
        "text_sha256": stable_hash(texts), "ids_sha256": stable_hash(ids),
    }
    if array_path.is_file() and metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        array = np.load(array_path)
        valid = all(metadata.get(key) == value for key, value in expected.items())
        valid = valid and array.ndim == 2 and array.shape[0] == len(texts)
        if valid:
            print(f"Кэш найден: {array_path.name}, shape={array.shape}")
            return array.astype(np.float32, copy=False)
        print(f"Кэш устарел, пересчитываю: {array_path.name}")

    previous_max_length = model.max_seq_length
    model.max_seq_length = int(max_length)
    started = time.perf_counter()
    try:
        with torch.inference_mode():
            array = model.encode(
                texts, batch_size=batch_size, show_progress_bar=True,
                convert_to_numpy=True, normalize_embeddings=True, device=DEVICE,
            ).astype(np.float32, copy=False)
    finally:
        model.max_seq_length = previous_max_length
    seconds = time.perf_counter() - started
    norms = np.linalg.norm(array[: min(5, len(array))], axis=1)
    assert np.allclose(norms, 1.0, atol=2e-3)
    metadata = {
        **expected, "embedding_dimension": int(array.shape[1]), "dtype": str(array.dtype),
        "encoding_seconds": seconds, "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    }
    np.save(array_path, array)
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Закодировано {len(texts):,} текстов за {seconds:.1f} с -> {array_path.name}")
    return array


chunk_ids = (chunks["article_id"].astype(str) + ":" + chunks["chunk_index"].astype(str)).tolist()
chunk_embeddings = encode_or_load(
    "bge_m3_chunk_embeddings_384_58", chunks["chunk_text"].astype(str).tolist(),
    chunk_ids, CHUNK_MAX_LENGTH, CHUNK_BATCH_SIZE,
)
calibration_query_embeddings = encode_or_load(
    "bge_m3_calibration_query_embeddings", calibration["query_norm"].astype(str).tolist(),
    calibration["query_id"].astype(np.int64).to_numpy().tolist(), QUERY_MAX_LENGTH, QUERY_BATCH_SIZE,
)
test_query_embeddings = encode_or_load(
    "bge_m3_test_query_embeddings", test["query_norm"].astype(str).tolist(),
    test["query_id"].astype(np.int64).to_numpy().tolist(), QUERY_MAX_LENGTH, QUERY_BATCH_SIZE,
)

model = None  # эмбеддер больше не нужен — освобождаю GPU-память
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

Эмбеддер загружен: BAAI/bge-m3


Batches:   0%|          | 0/316 [00:00<?, ?it/s]

Закодировано 3,787 текстов за 55.3 с -> bge_m3_chunk_embeddings_384_58.npy


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Закодировано 500 текстов за 0.6 с -> bge_m3_calibration_query_embeddings.npy
Кэш найден: bge_m3_test_query_embeddings.npy, shape=(500, 1024)


## 7. Скор статьи из скоров чанков и fusion

Близость «запрос–чанк» нужно превратить в близость «запрос-статья». Я беру **среднее двух
лучших чанков** статьи (top2-mean). Логика такая: максимум по чанкам поощряет случайные
выбросы одного фрагмента, а среднее по всем чанкам топит длинные статьи, где релевантен
только один раздел. Среднее двух лучших - компромисс, который на development оказался
лучше и того и другого. Заодно это защита от той самой гигантской статьи с сотнями чанков:
количество чанков само по себе преимущества не даёт.

Дальше **dense**: это семантический поиск по эмбеддингам, **fusion**: `0.75 × dense + 0.25 × BM25` (оба скора предварительно нормированы
per-query). Вес подобран на development перебором сетки.

In [10]:
chunk_indices = chunks["chunk_index"].astype(np.int64).to_numpy()
chunk_rows_by_article: dict[int, np.ndarray] = {
    int(article_id): group.index.to_numpy(dtype=np.int64)
    for article_id, group in chunks.groupby("article_id", sort=False)
}

calibration_chunk_scores = (calibration_query_embeddings @ chunk_embeddings.T).astype(np.float32)
test_chunk_scores = (test_query_embeddings @ chunk_embeddings.T).astype(np.float32)


def aggregate_chunk_top2_mean(query_scores: np.ndarray) -> np.ndarray:
    article_scores = np.empty((query_scores.shape[0], len(article_ids)), dtype=np.float32)
    for article_column, article_id in enumerate(tqdm(article_ids, desc="Агрегация чанков в статьи")):
        rows = chunk_rows_by_article[int(article_id)]
        values = query_scores[:, rows]
        n = min(2, values.shape[1])
        if n == 1:
            article_scores[:, article_column] = values.max(axis=1)
        else:
            top_values = np.partition(values, values.shape[1] - n, axis=1)[:, -n:]
            article_scores[:, article_column] = top_values.mean(axis=1)
    return article_scores


dense_calibration_scores = aggregate_chunk_top2_mean(calibration_chunk_scores)
dense_test_scores = aggregate_chunk_top2_mean(test_chunk_scores)

fusion_calibration_scores = (
    DENSE_WEIGHT * normalize_scores_per_query(dense_calibration_scores)
    + BM25_WEIGHT * normalize_scores_per_query(bm25_calibration_scores)
).astype(np.float32)
fusion_test_scores = (
    DENSE_WEIGHT * normalize_scores_per_query(dense_test_scores)
    + BM25_WEIGHT * normalize_scores_per_query(bm25_test_scores)
).astype(np.float32)

print("Матрицы скоров готовы:", fusion_calibration_scores.shape, fusion_test_scores.shape)

Агрегация чанков в статьи:   0%|          | 0/793 [00:00<?, ?it/s]

Агрегация чанков в статьи:   0%|          | 0/793 [00:00<?, ?it/s]

Матрицы скоров готовы: (500, 793) (500, 793)


## 8. Проверка качества retrieval на calibration

Смотрю MAP@10 трёх методов на development.
И отдельно ключевую диагностику этапа кандидатов — **oracle recall@50**: какая доля
релевантных статей вообще попала в топ-50 fusion. Это потолок всего пайплайна: статью,
которой нет среди кандидатов, следующие ступени уже не спасут.

In [11]:
def matrix_rankings(scores: np.ndarray, row_indices: np.ndarray, k: int = TOP_K) -> list[list[int]]:
    rankings = []
    for row in row_indices:
        order = np.lexsort((article_ids, -scores[row]))
        rankings.append([int(article_ids[i]) for i in order[:k]])
    return rankings


calibration_positions = np.arange(len(calibration))
dev_mask = calibration["split"].eq("development").to_numpy()
dev_positions = calibration_positions[dev_mask]
holdout_positions = calibration_positions[~dev_mask]
dev_frame = calibration.loc[dev_mask].reset_index(drop=True)
holdout_frame = calibration.loc[~dev_mask].reset_index(drop=True)

retrieval_report = []
for method_name, score_matrix in [
    ("bm25", bm25_calibration_scores),
    ("dense_top2_mean", dense_calibration_scores),
    ("fusion_0.75_0.25", fusion_calibration_scores),
]:
    metrics = evaluate_rankings(dev_frame, matrix_rankings(score_matrix, dev_positions))
    retrieval_report.append({"method": method_name, **metrics})

retrieval_report = pd.DataFrame(retrieval_report)
display(retrieval_report.round(4))

# Oracle recall@50: доля релевантных статей внутри топ-50 кандидатов fusion.
oracle_values = []
for row, ground_truth in zip(calibration_positions, calibration["ground_truth"]):
    order = np.lexsort((article_ids, -fusion_calibration_scores[row]))
    top50 = {int(article_ids[i]) for i in order[:CANDIDATE_N]}
    relevant = set(ground_truth)
    oracle_values.append(len(relevant & top50) / len(relevant))
print(f"\nOracle recall@{CANDIDATE_N} по всем 500 calibration-запросам: {np.mean(oracle_values):.4f}")

,method,MAP@10,Recall@10,MRR@10,HitRate@10
0,bm25,0.2562,0.5329,0.3076,0.6175
1,dense_top2_mean,0.4849,0.8123,0.5393,0.8825
2,fusion_0.75_0.25,0.4911,0.8306,0.5458,0.8900



Oracle recall@50 по всем 500 calibration-запросам: 0.9743


Fusion заметно сильнее каждого метода по отдельности, а oracle recall@50 около 0.97 -
то есть кандидатная ступень почти ничего не теряет, и узкое место задачи - ранжирование
внутри топ-50. Ровно этим занимаются шаги 3 и 4.

## 9. Сбор топ-50 кандидатов и выбор чанков для реранкера

Для каждого запроса сохраняю топ-50 статей по fusion со всеми скорами и рангами - они
станут признаками модели ранжирования в шаге 4. Плюс для каждого кандидата выбираю
**два ближайших к запросу чанка** - именно их будет читать cross-encoder в шаге 3
(отдавать ему статью целиком дорого и не нужно: важные для ответа места уже нашёл
векторный поиск).

In [12]:
def ranks_from_scores(scores: np.ndarray) -> np.ndarray:
    order = np.lexsort((article_ids, -scores))
    ranks = np.empty(len(article_ids), dtype=np.int32)
    ranks[order] = np.arange(1, len(article_ids) + 1)
    return ranks


def build_candidate_pairs(
    frame: pd.DataFrame,
    global_positions: np.ndarray,
    bm25_scores: np.ndarray,
    dense_scores: np.ndarray,
    fusion_scores: np.ndarray,
    chunk_scores: np.ndarray,
    split_name: str,
    with_labels: bool,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for local_index, global_index in enumerate(tqdm(global_positions, desc=f"Кандидаты ({split_name})")):
        query_row = frame.iloc[local_index]
        fusion_order = np.lexsort((article_ids, -fusion_scores[global_index]))
        bm25_ranks = ranks_from_scores(bm25_scores[global_index])
        dense_ranks = ranks_from_scores(dense_scores[global_index])
        fusion_ranks = ranks_from_scores(fusion_scores[global_index])

        for candidate_rank, article_column in enumerate(fusion_order[:CANDIDATE_N], start=1):
            article_id = int(article_ids[article_column])
            base = {
                "split": split_name,
                "query_id": int(query_row["query_id"]),
                "query_text": str(query_row["raw_query_text"]),
                "candidate_rank": int(candidate_rank),
                "article_id": article_id,
                "article_title": article_id_to_title[article_id],
                "bm25_score": float(bm25_scores[global_index, article_column]),
                "dense_score": float(dense_scores[global_index, article_column]),
                "fusion_score": float(fusion_scores[global_index, article_column]),
                "bm25_rank": int(bm25_ranks[article_column]),
                "dense_rank": int(dense_ranks[article_column]),
                "fusion_rank": int(fusion_ranks[article_column]),
                "global_query_index": int(global_index),
            }
            if with_labels:
                base["ground_truth"] = list(query_row["ground_truth"])

            # Два ближайших к запросу чанка этого кандидата — вход для cross-encoder.
            article_chunk_rows = chunk_rows_by_article[article_id]
            scores = chunk_scores[global_index, article_chunk_rows]
            n = min(2, len(article_chunk_rows))
            local_order = np.lexsort((chunk_indices[article_chunk_rows], -scores))[:n]
            for dense_chunk_rank, chunk_row_index in enumerate(article_chunk_rows[local_order], start=1):
                chunk_row = chunks.iloc[int(chunk_row_index)]
                rows.append({
                    **base,
                    "chunk_index": int(chunk_row["chunk_index"]),
                    "chunk_dense_rank": int(dense_chunk_rank),
                    "chunk_dense_score": float(chunk_scores[global_index, int(chunk_row_index)]),
                    "chunk_text": str(chunk_row["chunk_text"]),
                })

    pairs = pd.DataFrame(rows)
    assert pairs.groupby("query_id")["article_id"].nunique().eq(CANDIDATE_N).all()
    assert pairs.groupby(["query_id", "article_id"]).size().between(1, 2).all()
    assert not pairs.duplicated(["query_id", "article_id", "chunk_index"]).any()
    return pairs


dev_pairs = build_candidate_pairs(
    dev_frame, dev_positions, bm25_calibration_scores, dense_calibration_scores,
    fusion_calibration_scores, calibration_chunk_scores, "development", with_labels=True,
)
holdout_pairs = build_candidate_pairs(
    holdout_frame, holdout_positions, bm25_calibration_scores, dense_calibration_scores,
    fusion_calibration_scores, calibration_chunk_scores, "holdout", with_labels=True,
)
test_pairs = build_candidate_pairs(
    test.reset_index(drop=True), np.arange(len(test)), bm25_test_scores, dense_test_scores,
    fusion_test_scores, test_chunk_scores, "test", with_labels=False,
)

for name, pairs in [("development", dev_pairs), ("holdout", holdout_pairs), ("test", test_pairs)]:
    path = CANDIDATES_DIR / f"pairs_{name}_fusion_top_50.parquet"
    pairs.to_parquet(path, index=False)
    print(f"Сохранено: {path.name} — {len(pairs):,} пар (запрос, чанк кандидата)")

Кандидаты (development):   0%|          | 0/400 [00:00<?, ?it/s]

Кандидаты (holdout):   0%|          | 0/100 [00:00<?, ?it/s]

Кандидаты (test):   0%|          | 0/500 [00:00<?, ?it/s]

Сохранено: pairs_development_fusion_top_50.parquet — 37,390 пар (запрос, чанк кандидата)
Сохранено: pairs_holdout_fusion_top_50.parquet — 9,311 пар (запрос, чанк кандидата)
Сохранено: pairs_test_fusion_top_50.parquet — 46,542 пар (запрос, чанк кандидата)


## Выводы

1. На development: BM25 ≈ 0.26 MAP@10, векторный поиск по чанкам ≈ 0.49, fusion ≈ 0.50 -
   гибрид двух непохожих методов оправдал себя.
2. Oracle recall@50 ≈ 0.97: в топ-50 кандидатов попадают почти все правильные статьи,
   дальше задача сводится к точному ранжированию внутри этой полусотни.
3. Для каждого кандидата выбраны два ближайших к запросу чанка и сохранены все скоры
   и ранги - вход для шагов 3 и 4.

Дальше - шаг 3: cross-encoder перечитывает пары «запрос + чанк» и даёт куда более точный
скор релевантности.